# GAN on MNIST

Generative Adversarial Network trained from scratch on MNIST.

**What this notebook demonstrates:**
1. Alternating training loop: Discriminator step $\to$ Generator step
2. Non-saturating GAN loss (instead of the original $\log(1-D(G(z)))$)
3. Visualising generated samples as training progresses
4. Loss curves and the G $\leftrightarrow$ D dynamic
5. (Optional) Observing mode collapse

**Key mathematical idea:**
$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}} [\log D(x)] + \mathbb{E}_{z \sim p(z)}[\log(1 - D(G(z)))]$$

Implemented with the **non-saturating** trick for $G$:
$$\mathcal{L}_G = -\mathbb{E}_{z \sim p(z)}[\log D(G(z))]$$

The Generator maps noise $z \sim \mathcal{N}(0, I)$ to images via transposed convolutions;
the Discriminator classifies images as real or fake via standard convolutions.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Project imports
p = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if p not in sys.path:
    sys.path.insert(0, p)

from core.gen import Discriminator, Generator

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 128
EPOCHS = 50
LR = 2e-4  # DCGAN recommends 2e-4 for Adam
LATENT_DIM = 100  # noise dimension (DCGAN standard)
BETAS = (0.5, 0.999)  # Adam betas: DCGAN uses beta1=0.5

N_CRITIC = 1  # D updates per G update (1 = standard GAN)

torch.manual_seed(42)
np.random.seed(42)
print(f"Device: {DEVICE}")

### Data — MNIST (normalised to $[-1, 1]$)

GAN's Generator outputs images via $	anh$ in $(-1, 1)$. The data must be
scaled to the same range so the Discriminator can compare them fairly.


In [ ]:
DATA_ROOT = (
    Path.cwd().parent / "assets" if Path.cwd().name == "apps" else Path.cwd() / "assets"
)

# ToTensor() scales to [0, 1]; Normalise(mean=0.5, std=0.5) shifts to [-1, 1]
# matching the Generator's Tanh output range.
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(0.5, 0.5),
    ]
)

train_set = datasets.MNIST(DATA_ROOT, train=True, download=True, transform=transform)
train_loader = DataLoader(
    train_set, BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True
)  # drop_last for consistent batch sizes

print(f"Train: {len(train_set)}")

In [ ]:
# Sample a few training images
# NOTE: imgs are in [-1, 1]; rescale to [0, 1] for display via (x + 1) / 2
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow((imgs[i].squeeze() + 1) / 2, cmap="gray")
    ax.set_title(f"label={labels[i].item()}", fontsize=9)
    ax.axis("off")
fig.suptitle("MNIST samples (normalised to [-1, 1])")

### Model — Generator + Discriminator

Both follow the DCGAN architecture:

- **Generator**: Noise $z$ → Linear + reshape → ConvTranspose2d × 2 ($7\to14\to28$) → Tanh
- **Discriminator**: Image → Conv2d × 3 ($28\to14\to7\to3$) → Linear → logit


In [ ]:
generator = Generator(
    latent_dim=LATENT_DIM,
    out_channels=1,
    img_size=28,
    ngf=64,
).to(DEVICE)

discriminator = Discriminator(
    in_channels=1,
    img_size=28,
    ndf=64,
).to(DEVICE)

g_params = sum(p.numel() for p in generator.parameters())
d_params = sum(p.numel() for p in discriminator.parameters())
print(f"Generator: {g_params / 1e3:.1f}K parameters")
print(f"Discriminator: {d_params / 1e3:.1f}K parameters")

### Weight Initialisation

DCGAN recommends specific weight initialisation:
$\mathcal{W} \sim \mathcal{N}(0, 0.02)$ applied to all Conv / ConvTranspose / Linear layers.
This prevents vanishing gradients early in training.


In [ ]:
# DCGAN-style weight initialisation:
#   Conv2d, ConvTranspose2d, Linear → normal_(0, 0.02)
#   BatchNorm2d → weight=1, bias=0


def weights_init(m):
    """DCGAN weight initialisation: N(0, 0.02) for all conv/linear layers."""
    classname = m.__class__.__name__
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)


generator.apply(weights_init)
discriminator.apply(weights_init)
print("Weights initialised (normal 0, 0.02)")

### Loss & Optimisers

We use **BCEWithLogitsLoss** which combines Sigmoid + BCE in one numerically
stable operation — the Discriminator outputs raw logits (not probabilities).

Two optimisers: one for $G$, one for $D$.  DCGAN recommends Adam with $\beta_1=0.5$
(lower than the default 0.9) to dampen oscillations in the adversarial game.


In [ ]:
criterion = nn.BCEWithLogitsLoss()

optimizer_G = optim.Adam(generator.parameters(), lr=LR, betas=BETAS)
optimizer_D = optim.Adam(discriminator.parameters(), lr=LR, betas=BETAS)

### Training Loop

At each step:
1. **D real step**: real image $x \to D(x)$; target $= 1$ → $\mathcal{L}_{D,\text{real}}$
2. **D fake step**: noise $z \to G(z) \to D(G(z))$; target $= 0$ → $\mathcal{L}_{D,\text{fake}}$
3. **G step**: noise $z \to G(z) \to D(G(z))$; target $= 1$ (non-saturating) → $\mathcal{L}_G$

$$\mathcal{L}_D = \frac12\big[\text{BCE}(D(x), 1) + \text{BCE}(D(G(z)), 0)\big]$$
$$\mathcal{L}_G = \text{BCE}(D(G(z)), 1)$$

We alternate: update $D$ once, then $G$ once (unless $N_{\text{critic}} > 1$).


In [ ]:
# Training loop with alternating D and G updates.
#
#   D step: real=D(x) → BCE(real, 0.9)  +  G(z)→detach→D(fake) → BCE(fake, 0.0)
#   G step: G(z) → D(fake) → BCE(fake, 0.9)   ← non-saturating!
#
# NOTE: Detach fake images in D step so gradients don't flow into G.
# NOTE: Label smoothing (real=0.9) prevents D from being overconfident.

fixed_noise = torch.randn(
    64, LATENT_DIM, device=DEVICE
)  # fixed seed for visualising progress

g_losses = []
d_losses = []
snapshots = []  # (epoch, generated_images_grid) for epoch progression
snapshot_epochs = {1, 5, 10, 25, 50}  # which epochs to snapshot

for epoch in range(EPOCHS):
    epoch_g_loss = 0.0
    epoch_d_loss = 0.0
    n_batches = 0

    for real_imgs, _ in train_loader:
        real_imgs = real_imgs.to(DEVICE)
        batch_size = real_imgs.size(0)

        # Labels with one-sided label smoothing.
        # D returns shape (batch,) so labels are 1D too.
        real_label = torch.full((batch_size,), 0.9, device=DEVICE)
        fake_label = torch.full((batch_size,), 0.0, device=DEVICE)

        # ═══════════════ D step ═══════════════
        optimizer_D.zero_grad()

        # Real images: D(x) should → 1 (real)
        output_real = discriminator(real_imgs)
        loss_d_real = criterion(output_real, real_label)

        # Fake images: D(G(z)) should → 0 (fake)
        noise = torch.randn(batch_size, LATENT_DIM, device=DEVICE)
        fake_imgs = generator(noise)
        output_fake = discriminator(fake_imgs.detach())  # detach: no gradient to G
        loss_d_fake = criterion(output_fake, fake_label)

        loss_D = (loss_d_real + loss_d_fake) / 2
        loss_D.backward()
        optimizer_D.step()

        # ═══════════════ G step ═══════════════
        optimizer_G.zero_grad()

        # G(z) → D(G(z)): target = 1 (non-saturating!)
        noise = torch.randn(batch_size, LATENT_DIM, device=DEVICE)
        fake_imgs = generator(noise)
        output = discriminator(fake_imgs)
        loss_G = criterion(output, real_label)  # non-saturating: target=real!

        loss_G.backward()
        optimizer_G.step()

        epoch_g_loss += loss_G.item() * batch_size
        epoch_d_loss += loss_D.item() * batch_size
        n_batches += batch_size

    g_losses.append(epoch_g_loss / n_batches)
    d_losses.append(epoch_d_loss / n_batches)

    # Snapshot for epoch progression visualisation
    if (epoch + 1) in snapshot_epochs:
        with torch.no_grad():
            snapshots.append((epoch + 1, generator(fixed_noise).cpu()))

    print(
        f"Epoch {epoch + 1:3d}/{EPOCHS} | "
        f"D loss {d_losses[-1]:.4f} | G loss {g_losses[-1]:.4f}"
    )

### 1. Loss Curves

GAN loss curves tell a story:

- **D loss around $\log(2) \approx 0.693$** — when D can't distinguish real from fake (optimal)
- **D loss $\ll 0.693$** — D is dominating (generator is weak)
- **D loss $\gg 0.693$** — G is winning (D can't tell real from fake)
- **Oscillations** — normal! The two players are constantly adapting to each other


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(g_losses, label="Generator loss", alpha=0.8)
ax.plot(d_losses, label="Discriminator loss", alpha=0.8)
ax.axhline(np.log(2), color="gray", ls="--", alpha=0.5, label="D random guess = log(2)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("GAN Training: G vs D Loss")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

### 2. Generated Samples (Epoch Progression)

Visualise how the Generator improves over time — from noise to digits.


In [ ]:
# Show generated samples at different training stages.
# Each snapshot uses the same fixed noise seed, so the progression is directly comparable.

n_snapshots = len(snapshots)
fig, axes = plt.subplots(2, n_snapshots, figsize=(n_snapshots * 2.5, 5))

for col, (ep, imgs) in enumerate(snapshots):
    # First row: one sample enlarged
    axes[0, col].imshow(imgs[0].squeeze(), cmap="gray", vmin=-1, vmax=1)
    axes[0, col].set_title(f"Epoch {ep}", fontsize=10)
    axes[0, col].axis("off")

    # Second row: create a 4×4 grid from the first 16 images
    grid_img = imgs[:16]  # (16, 1, 28, 28)
    # Arrange into a 4×4 grid manually
    grid_rows = []
    for r in range(4):
        row = torch.cat([grid_img[r * 4 + c] for c in range(4)], dim=2)
        grid_rows.append(row)
    grid = torch.cat(grid_rows, dim=1).squeeze()  # (112, 112)
    axes[1, col].imshow(grid, cmap="gray", vmin=-1, vmax=1)
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Single sample", fontsize=10)
axes[1, 0].set_ylabel("16 samples", fontsize=10)
fig.suptitle("Generator progression over training (same noise seed)", fontsize=13)
plt.tight_layout()

### 3. Final Generated Samples

A $4\times 4$ grid of digits generated from random noise after training completes.


In [ ]:
@torch.no_grad()
def show_generated(generator, n=16):
    """Generate and display n images from random noise."""
    generator.eval()
    z = torch.randn(n, LATENT_DIM, device=DEVICE)
    samples = generator(z).cpu()  # (n, 1, 28, 28)

    # Plot as 4×4 grid
    fig, axes = plt.subplots(4, 4, figsize=(5, 5))
    for i, ax in enumerate(axes.flat):
        ax.imshow(samples[i].squeeze(), cmap="gray", vmin=-1, vmax=1)
        ax.axis("off")
    fig.suptitle("Random generation: z ~ N(0, I) → G(z)", fontsize=12)
    plt.tight_layout()


show_generated(generator)

### 4. Discriminator Confidence

How confident is $D$ about real vs generated images?
If $D$ assigns very high confidence to all real images and very low to all
fakes, the training may be unbalanced.


In [ ]:
# Compare Discriminator confidence on real vs generated images.
# Overlapping distributions → G is fooling D well.
# Complete separation → one player is dominating.


@torch.no_grad()
def d_confidence_histogram(generator, discriminator, loader, n=512):
    """Plot D(x) vs D(G(z)) confidence distributions."""
    generator.eval()
    discriminator.eval()

    # Get real images
    real_imgs, _ = next(iter(loader))
    real_imgs = real_imgs[: n // 2].to(DEVICE)

    # Generate fake images
    z = torch.randn(n // 2, LATENT_DIM, device=DEVICE)
    fake_imgs = generator(z)

    # D scores (apply sigmoid to convert logits → [0, 1] confidence)
    d_real = torch.sigmoid(discriminator(real_imgs)).cpu().numpy()
    d_fake = torch.sigmoid(discriminator(fake_imgs)).cpu().numpy()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(
        d_real,
        bins=30,
        alpha=0.6,
        label=f"D(x) real  (mean={d_real.mean():.3f})",
        color="tab:blue",
        density=True,
    )
    ax.hist(
        d_fake,
        bins=30,
        alpha=0.6,
        label=f"D(G(z)) fake (mean={d_fake.mean():.3f})",
        color="tab:orange",
        density=True,
    )
    ax.axvline(0.5, color="gray", ls="--", alpha=0.5, label="D=0.5 (random guess)")
    ax.set_xlabel("Discriminator confidence")
    ax.set_ylabel("Density")
    ax.set_title("D Confidence: real vs generated images")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()


d_confidence_histogram(generator, discriminator, train_loader)

### 5. (Optional) Mode Collapse Detection

**Mode collapse** is the most common GAN failure mode: the Generator
discovers a few "tricks" that fool $D$ and outputs the same digit
over and over regardless of the noise $z$.

To detect it:
- Generate many samples and check the diversity of output classes
- Use a pre-trained MNIST classifier (or just visual inspection)
- Compute the variance of generated images across samples — low
  variance suggests collapse


In [ ]:
# Mode collapse detection: check diversity of generated samples.
#
# Mode collapse = Generator outputs the same (or very similar) image
# regardless of the input noise.  We detect this by:
#   1. Generating a batch of 64 images
#   2. Computing inter-sample pixel variance (low → possible collapse)
#   3. Visual inspection of the batch


@torch.no_grad()
def check_mode_collapse(generator, n=64):
    """Detect mode collapse by measuring diversity across generated samples."""
    generator.eval()
    z = torch.randn(n, LATENT_DIM, device=DEVICE)
    samples = generator(z)  # (n, 1, 28, 28)

    # Pixel-wise std across samples: average of per-pixel standard deviations
    # Low values → all generated images look similar → possible mode collapse
    pixel_std = samples.std(dim=0).mean().item()

    # Global std of all pixel values across the batch
    global_std = samples.std().item()

    print(f"Avg pixel-wise std across {n} samples: {pixel_std:.4f}")
    print(f"Global std across {n} samples:              {global_std:.4f}")
    print()

    # Visualise: show the 64 images in an 8×8 grid
    fig, axes = plt.subplots(8, 8, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        ax.imshow(samples[i].cpu().squeeze(), cmap="gray", vmin=-1, vmax=1)
        ax.axis("off")
    fig.suptitle(
        f"64 generated samples — "
        f"pixel_std={pixel_std:.3f}, global_std={global_std:.3f}",
        fontsize=12,
    )
    plt.tight_layout()

    # Rule of thumb: pixel_std < 0.1 suggests possible mode collapse
    if pixel_std < 0.1:
        print("⚠️  WARNING: Low pixel diversity — possible mode collapse!")
        print("   Try: label smoothing, more noise dims, or different LR.")
    else:
        print("✅ Samples appear diverse (no strong evidence of mode collapse).")


check_mode_collapse(generator)

### Summary

**What you should take away:**

1. **Adversarial training** — two networks compete: $G$ learns to fool $D$, $D$ learns to catch $G$
2. **Non-saturating loss** — flipping $G$'s target from 0→1 fixes early vanishing gradients
3. **Training instability** — GAN is famously hard to train; loss oscillations are expected
4. **Mode collapse** — the Generator may "cheat" by only producing a few outputs

**Troubleshooting tips if training diverges:**
- Reduce learning rate (try 1e-4 instead of 2e-4)
- Increase Generator updates per D update (try $N_{\text{critic}} = 3$)
- Add label smoothing (real = 0.9, fake = 0.1)
- Try different architectures: more/fewer filters, deeper/shallower nets